# BIRD Mini-Dev: Schema Pruning trên Kaggle

Notebook có hai giai đoạn: (1) retrieval trên 500 câu chính thức, không cần GPU/SQLite; (2) đối chiếu full/BM25/hybrid khi đã gắn đúng SQLite và model.

Chọn accelerator có sẵn trong Notebook Settings. Bật Internet nếu tải dependency/model/data; phiên offline cần gắn sẵn các artifact. Mọi kết quả được ghi dưới `/kaggle/working`.

Gold SQL chỉ được dùng ở evaluator. Nhãn schema suy ra từ gold SQL; đây không phải annotation schema linking chính thức. EX của dự án dùng bộ so sánh riêng, cần scorer chính thức để báo cáo BIRD EX/R-VES.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

# Sửa đường dẫn này theo dataset chứa source code bạn đã thêm vào Kaggle.
REPO_INPUT = Path("/kaggle/input/slm-prj/SLM-PRJ")
WORK = Path("/kaggle/working/SLM-PRJ")
assert (REPO_INPUT / "core/pipeline.py").is_file(), f"Sửa REPO_INPUT: {REPO_INPUT}"
shutil.copytree(REPO_INPUT, WORK, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns(".git", ".venv", "__pycache__", ".cache", "results", "artifacts"))
os.chdir(WORK)
sys.path.insert(0, str(WORK))
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True, cwd=WORK)
print(WORK)


In [ ]:
# Kaggle thường có sẵn PyTorch; giữ build GPU của môi trường.
# Nếu Internet tắt, chuẩn bị wheelhouse và thay lệnh này bằng pip --no-index --find-links.
run("-m", "pip", "install", "-r", "requirements.txt", "-r", "requirements-model.txt")
run("-m", "pytest", "-q")


## 1. Tải metadata chính thức đã ghim phiên bản

Ba tệp khoảng 538 KiB, gồm 500 câu, schema 11 DB và manifest. Script kiểm tra SHA-256. Không tải SQLite hoặc trọng số model.

Nếu offline, thay `DATA` bằng thư mục metadata đã gắn sẵn và bỏ lệnh tải. Dataset revision được ghim trong script; không trộn với archive 2025 hoặc bộ dev khác.


In [ ]:
DATA = WORK / "data/bird_minidev"
run("scripts/download_bird_minidev.py", "--output", DATA)
QUESTIONS = DATA / "mini_dev_sqlite.jsonl"
SCHEMAS = DATA / "schemas.json"
print(json.loads((DATA / "provenance.json").read_text())["revision"])


In [ ]:
import pandas as pd
# Baseline và BM25 chỉ chạy CPU. Token mặc định là proxy regex, được gắn nhãn rõ.
from evaluation.evaluate_schema_retrieval import evaluate as evaluate_retrieval
retrieval_reports = []
for mode, tables, columns in [("full", 4, 8), ("bm25", 4, 8), ("bm25", 6, 16), ("bm25", 8, 24), ("bm25", 8, 48)]:
    label = "full" if mode == "full" else f"{mode}_t{tables}_c{columns}"
    report = evaluate_retrieval(QUESTIONS, SCHEMAS, WORK / "results/retrieval" / label,
                               mode=mode, top_k_tables=tables, top_k_columns=columns)
    retrieval_reports.append({"label": label, **report["summary"]})
pd.DataFrame(retrieval_reports)[["label", "questions", "label_errors", "column_recall_macro",
    "all_references_covered", "prompt_token_reduction_micro", "retrieval_ms_p95"]]


## 2. Đếm token thật và thử hybrid (tùy chọn)

Dùng cùng tokenizer với mô hình đánh giá. Token ở bước retrieval là văn bản prompt thô; backend khi sinh SQL ghi riêng số token sau chat template.

Hybrid tải encoder riêng; cache vector lưu dưới `/kaggle/working`. Với 11 DB nhỏ, cosine trên ma trận là đủ, chưa cần dịch vụ vector database. Chạy lượt đầu xây index và lượt thứ hai để so sánh cache; giữ log cả hai.


In [ ]:
from transformers import AutoTokenizer
from core.config import load_config
RUN_SEMANTIC = False  # Bật khi muốn chạy thêm hybrid.
MODEL_ID = "cycloneboy/SLM-SQL-1.5B"  # Hoặc đường dẫn model đã gắn sẵn.
ALLOW_DOWNLOADS = True  # False khi dùng cache/model offline.
config = load_config()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=not ALLOW_DOWNLOADS)
for mode in (["full", "bm25", "hybrid"] if RUN_SEMANTIC else ["full", "bm25"]):
    if mode == "hybrid":
        run("-m", "pip", "install", "-r", "requirements-retrieval.txt")
    report = evaluate_retrieval(QUESTIONS, SCHEMAS, WORK / "results/retrieval_tokens" / mode,
        mode=mode, top_k_tables=8, top_k_columns=48, tokenizer=tokenizer, tokenizer_name=MODEL_ID,
        retriever_options={"local_files_only": not ALLOW_DOWNLOADS,
            "device": "cpu", "cache_dir": str(WORK / ".cache/schema_embeddings")})
    print(mode, report["summary"])


## 3. Gắn SQLite cùng phiên bản và chuyển dữ liệu

Thêm package SQLite chính thức từ hướng dẫn BIRD, giải nén thành `<DB_ROOT>/<db_id>/<db_id>.sqlite`. Chỉ chạy tiếp khi đã đối chiếu phiên bản SQLite với manifest/questions. Metadata mới và DB archive cũ có thể không khớp; không coi lỗi gold SQL là lỗi model.

Để chạy BIRD dev, thay QUESTIONS bằng `dev.json` tương ứng và dùng đúng `dev_databases`; phần retrieval hỗ trợ `dev_tables.json` theo định dạng BIRD/Spider.


In [ ]:
RUN_GENERATION = False  # Bật sau khi đã gắn SQLite và model.
DB_ROOT = Path("/kaggle/input/bird-sqlite/dev_databases")  # Sửa theo dataset đã gắn.
DATASET = WORK / "data/benchmarks/bird_eval.json"
if RUN_GENERATION:
    run("scripts/import_bird.py", "--questions", QUESTIONS, "--database-root", DB_ROOT, "--output", DATASET)


## 4. So sánh full schema và pruning trên cùng câu hỏi

Mặc định smoke test 10 câu, sau đó `LIMIT = None` để đánh giá đủ 500. Greedy, batch 1, tắt correction, không sample row giúp cô lập tác động của pruning. Bộ execution có giới hạn hữu hạn; nếu gold SQL vượt giới hạn, điều chỉnh cùng một cấu hình cho mọi arm rồi chạy lại.

Lượt warm-up không đưa vào báo cáo. Chạy cùng model/hardware, ghi model revision và `pip freeze`. Thời gian có cả schema inspection/retrieval/generation; không suy ra mức giảm thời gian hoặc VRAM trực tiếp từ giảm token. Peak được ghi cho từng GPU; tổng các peak theo GPU không nhất thiết cùng thời điểm. Sau thử nghiệm exploratory, chốt cấu hình và đánh giá tập held-out khác (ví dụ BIRD dev), tránh chọn ngân sách tốt nhất trên Mini-Dev rồi gọi đó là kết quả tổng quát hóa.


In [ ]:
if RUN_GENERATION:
    import copy
    import torch
    from core.model_backend import create_backend
    from evaluation.evaluate import evaluate as evaluate_sql
    from core.pipeline import Pipeline
    assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings"
    config["model"].update(backend="huggingface", model_id=MODEL_ID, device="cuda",
                            local_files_only=not ALLOW_DOWNLOADS)
    config["sampling"]["mode"] = "greedy"
    config["generation"].update(batch_size=1, max_new_tokens=512)
    config["correction"]["enabled"] = False
    config["schema"]["sample_rows"] = 0
    config["retrieval"].update(top_k_tables=8, top_k_columns=48, local_files_only=not ALLOW_DOWNLOADS,
                                cache_dir=str(WORK / ".cache/schema_embeddings"))
    # BIRD query có thể dài; các ngưỡng áp dụng giống nhau ở mọi arm.
    config["execution"].update(timeout_seconds=30, max_steps=50_000_000,
                                max_result_rows=100_000, max_result_bytes=64*1024*1024)
    backend = create_backend(config)
    first = json.loads(DATASET.read_text())[0]
    first_db = (DATASET.parent / first["db_path"]).resolve()
    Pipeline(config, backend).run(first_db, first["question"], first.get("evidence", ""))
    LIMIT = 10  # None = toàn bộ câu hỏi.
    summaries = []
    for mode in (["full", "bm25", "hybrid"] if RUN_SEMANTIC else ["full", "bm25"]):
        arm = copy.deepcopy(config)
        arm["validation"]["schema_selection"] = mode != "full"
        if mode != "full":
            arm["retrieval"]["mode"] = mode
        output = WORK / "results/generation" / mode
        summary = evaluate_sql(arm, DATASET, first_db, output, LIMIT, backend=backend, profile_cuda=True)
        summaries.append(summary)
    display(pd.DataFrame(summaries)[["schema_mode", "questions", "execution_accuracy",
        "average_latency_ms", "mean_model_input_tokens_per_call", "context_overflows",
        "peak_cuda_allocated_mb_sum_device_peaks"]])
    with (WORK / "results/pip-freeze.txt").open("w") as handle:
        subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True)
    print("Model commit:", getattr(backend.model.config, "_commit_hash", None))


## 5. Lưu kết quả và kiểm tra scorer chính thức

`summary.json` chứa thống kê, `predictions.jsonl` chứa từng câu với SQL chọn, lỗi, input/output token và peak CUDA memory. Retrieval có `details.jsonl` để xem bảng/cột bị mất. Giữ cả failed/context overflow trong mẫu số.

Bộ so sánh execution nội bộ giữ duplicate và xét ORDER BY/LIMIT; nó không hoàn toàn giống scorer BIRD. Trước khi công bố điểm benchmark, xuất prediction theo hướng dẫn evaluator chính thức ở [BIRD Mini-Dev](https://github.com/bird-bench/mini_dev), chạy đúng SQLite/GT cùng phiên bản. Không dùng kết quả proxy token hoặc mock để báo EX/VES.

Exporter có sẵn: `python scripts/export_bird_predictions.py --dataset <subset-evaluated.json> --predictions <predictions.jsonl> --output <official-dir>`. Xem hướng dẫn `docs/SCHEMA_PRUNING_BIRD_KAGGLE.md` cho lệnh EX đã ghim commit, JSONL difficulty và yêu cầu DB cùng phiên bản.


In [ ]:
archive = shutil.make_archive("/kaggle/working/schema_pruning_results", "zip", WORK / "results")
print(archive)
